In [ ]:
import sys, os
sys.path.append("../")

import jax
jax.config.update("jax_enable_x64", True)

from qd_solve import *
from qd_solve.eig import *
from qd_solve.exp import *
from qd_solve.operator import *
from qd_solve.split import *
from qd_solve.system import *
from qd_solve.spaces.pseudospectral import *

from miscutils.plot import animate

import jax.numpy as jnp
import diffrax

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Image

In [ ]:
x0 = -10
xf = 10
num_steps = 100
num_modes = 100
hilbert_space = PseudoSpectral(x0, xf, num_steps, num_modes)

potential = lambda x: 0.5 * x ** 2
V = PseudoSpectralPotentialEnergy(potential)
T = -0.5 * PseudoSpectralLaplacian()

dt = 0.01

In [ ]:
key = jax.random.key(0)

def test_order(dt):
    exp = ScaleSquareExponentiator(V, hilbert_space, dt, m=10)
    V_approx = V.set_exponentiator(exp)
    y_vals = jax.random.normal(key, shape=(num_steps,), dtype=jnp.array(1j).dtype)
    y = hilbert_space.from_values(y_vals)
    result1 = V_approx.exp(-1j * dt, y).coeffs
    result2 = V.exp(-1j * dt, y).coeffs
    return jnp.mean(jnp.linalg.norm(result1 - result2, axis=0))

dt_range = 10 ** jnp.linspace(-3, 0, 100)
results = jnp.array([test_order(dt) for dt in dt_range])

(results[-1] - results[0]) / (dt_range[-1] - dt_range[0])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

ax.loglog(dt_range, results)

plt.show()
plt.close(fig)